# LAB: Agents avec LangGraph (Advanced)
## Building Custom Agents with LangGraph
### Master SDIA - Prof. RETAL SARA

Ce notebook couvre la création d'agents avancés **directement avec LangGraph** au lieu d'utiliser `create_agent()`.

## 📚 Pourquoi LangGraph au lieu de create_agent()?

### **create_agent() (Simple)**
- ✅ Rapide à mettre en place
- ✅ Agent prêt à l'emploi
- ✅ Boucle de raisonnement construite
- ❌ Moins de contrôle
- ❌ Moins flexible
- ❌ Difficile à personnaliser

### **LangGraph (Avancé)**
- ✅ Contrôle complet
- ✅ Highly customizable
- ✅ Agents complexes
- ✅ State management granulaire
- ✅ Streaming native
- ✅ HITL intégré
- ❌ Plus de code
- ❌ Courbe d'apprentissage

### Quand utiliser LangGraph?
- 🎯 Agents avec logique complexe
- 🎯 Workflows multi-étapes
- 🎯 State management avancé
- 🎯 Human-in-the-loop
- 🎯 Production systems
- 🎯 RAG agents avancés

## ⚙️ SETUP

In [ ]:
print("""
🚀 INSTALLATION LANGGRAPH
="*70

uv add langchain langchain-ollama langchain-core langgraph python-dotenv

Packages clés:
  • langgraph - Graph workflow framework
  • StateGraph - Pour construire le graphe
  • START, END - Nœuds spéciaux
  • TypedDict - État typé
""")

## 📋 PARTIE 1: Setup des Tools

In [ ]:
from langchain.tools import tool
from langchain_ollama import ChatOllama

print("\n📋 SETUP DES TOOLS\n")
print("="*70)

# Initialiser le modèle
model = ChatOllama(
    model="llama3.2:3b",
    temperature=0
)

print("✅ Modèle LLM configuré: llama3.2:3b")
print()

# Définir les tools
@tool
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b

@tool
def divide(a: int, b: int) -> float:
    """Divide two integers."""
    if b == 0:
        return "Error: Division by zero"
    return a / b

tools = [add, multiply, divide]
tools_by_name = {t.name: t for t in tools}

print("✅ Tools créés:")
for t in tools:
    print(f"   • {t.name}: {t.description}")
print()

# Binder les tools au modèle
model_with_tools = model.bind_tools(tools)
print("✅ Tools bindés au modèle")
print("   Le modèle peut maintenant décider d'utiliser les tools")

## 🤖 PARTIE 2: Agent comme nœud de LangGraph

In [ ]:
print("""
🤖 AGENT AVEC LANGGRAPH
="*70

Architecture:

    START
      ↓
    llm_call (Agent décide d'utiliser un tool ou non)
      ↓
   should_continue? (Conditional routing)
      ├─ YES → tool_node (Exécute le tool)
      │          ↓
      │        llm_call (Utilise le résultat pour répondre)
      │          ↓
      └─ NO → END (Réponse finale)

État (AgentState):
  • messages: Historique complet des messages
  • llm_calls: Compteur des appels LLM

Flux:
  1. Agent reçoit question
  2. LLM décide: Utiliser tool? Oui/Non
  3. Si OUI: Exécute le tool
  4. Réponse finale
""")

print("\nCode pour créer le graphe:\n")
print("""
from typing_extensions import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph, START, END
from langchain.messages import SystemMessage, ToolMessage, HumanMessage

class AgentState(TypedDict):
    messages: Annotated[list, add]  # add est le reducer
    llm_calls: int

def llm_call(state: AgentState):
    response = model_with_tools.invoke(
        [SystemMessage(content="Tu es un assistant mathématique.")] + 
        state["messages"]
    )
    return {
        "messages": [response],
        "llm_calls": state.get("llm_calls", 0) + 1,
    }

def tool_node(state: AgentState):
    last = state["messages"][-1]
    results = []
    for call in last.tool_calls:
        tool = tools_by_name[call["name"]]
        observation = tool.invoke(call["args"])
        results.append(ToolMessage(
            content=str(observation),
            tool_call_id=call["id"]
        ))
    return {"messages": results}

def should_continue(state: AgentState):
    last = state["messages"][-1]
    if getattr(last, "tool_calls", None):
        return "tool_node" if last.tool_calls else END
    return END

builder = StateGraph(AgentState)
builder.add_node("llm_call", llm_call)
builder.add_node("tool_node", tool_node)
builder.add_edge(START, "llm_call")
builder.add_conditional_edges("llm_call", should_continue, 
                              ["tool_node", END])
builder.add_edge("tool_node", "llm_call")

agent = builder.compile()
""")

## 🔄 PARTIE 3: Streaming

In [ ]:
print("""
🔄 STREAMING AVEC LANGGRAPH
="*70

LangGraph supporte plusieurs modes de streaming:

1️⃣ stream_mode="updates"
   └─ Affiche chaque modification du state node par node
   └─ Utile pour voir l'exécution en temps réel
   └─ Résultat: {'node_name': {'state_field': value}}

2️⃣ stream_mode="messages"
   └─ Affiche les tokens du LLM (streaming natif)
   └─ Utile pour les réponses en temps réel
   └─ Résultat: Stream de tokens

3️⃣ stream_mode="values"
   └─ Affiche l'état complet à chaque étape
   └─ Utile pour le debugging

Exemples:

```python
# Mode updates (node par node)
for chunk in agent.stream(
    {"messages": [HumanMessage(content="Add 3 and 4.")], "llm_calls": 0},
    stream_mode="updates"
):
    print(chunk)
    # → {'llm_call': {'messages': [...], 'llm_calls': 1}}
    # → {'tool_node': {'messages': [...]}}

# Mode messages (LLM tokens)
for message_chunk in agent.stream(
    {"messages": [HumanMessage(content="Add 3 and 4.")], "llm_calls": 0},
    stream_mode="messages"
):
    if hasattr(message_chunk, 'content'):
        print(message_chunk.content, end="", flush=True)

# Afficher l'historique complet
result = agent.invoke(
    {"messages": [HumanMessage(content="Add 3 and 4.")], "llm_calls": 0}
)
for message in result["messages"]:
    message.pretty_print()
```
""")

## ✅ PARTIE 4: HITL avec LangGraph

In [ ]:
print("""
✅ HITL AVEC LANGGRAPH
="*70

Intégrer HITL (Human-In-The-Loop) dans un agent LangGraph:

```python
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver

def approve_node(state: AgentState) -> Command:
    """Node qui demande approbation avant exécution de tools."""
    last = state["messages"][-1]
    
    # Arrêter et demander approbation
    decision = interrupt({
        "question": "Approve tool execution?",
        "tool_calls": last.tool_calls
    })
    
    # Si approuvé, continuer avec tool_node
    # Si rejeté, aller à END
    return Command(
        goto="tool_node" if decision else END
    )

builder = StateGraph(AgentState)
builder.add_node("llm_call", llm_call)
builder.add_node("approve", approve_node)  # ← HITL ici
builder.add_node("tool_node", tool_node)

checkpointer = InMemorySaver()

builder.add_edge(START, "llm_call")
builder.add_conditional_edges(
    "llm_call",
    should_continue,
    ["approve", END]  # Approuver avant tools
)
builder.add_edge("tool_node", "llm_call")

agent = builder.compile(checkpointer=checkpointer)

# Utilisation
config = {"configurable": {"thread_id": "user-123"}}

result = agent.invoke(
    {"messages": [HumanMessage(content="Add 3 and 4.")], "llm_calls": 0},
    config=config
)

# Agent s'arrête en attente d'approbation
print(result["__interrupt__"][0].value)

# Reprendre avec approbation
resume = agent.invoke(Command(resume=True), config=config)
print("Done:", resume["messages"][-1])
```

Flux HITL:
    1. Agent propose tool
    2. approve_node() arrête l'exécution
    3. Humain voit: {"question": "Approve?", "tool_calls": [...]}
    4. Humain envoie: Command(resume=True)
    5. Agent exécute le tool
    6. Résultat final
""")

## 📊 Résumé: LangGraph vs create_agent()

In [ ]:
print("""
📊 COMPARAISON
="*70

┌─────────────────────┬──────────────────┬──────────────────┐
│ Aspect              │ create_agent()   │ LangGraph        │
├─────────────────────┼──────────────────┼──────────────────┤
│ Lignes de code      │ ~10 lignes       │ ~50 lignes       │
│ Contrôle            │ Bas              │ Total            │
│ Flexibilité         │ Basse            │ Haute            │
│ HITL                │ Difficile        │ Intégré          │
│ State management    │ Simple           │ Avancé           │
│ Streaming           │ Non natif        │ Natif            │
│ Courbe d'apprentissage │ Faible        │ Moyenne          │
│ Production ready    │ Oui              │ Oui              │
└─────────────────────┴──────────────────┴──────────────────┘

Quand utiliser quoi?

create_agent():
  ✓ Prototypes rapides
  ✓ Agents simples
  ✓ POC et démos
  ✓ Apprenants LangChain

LangGraph:
  ✓ Agents avancés
  ✓ HITL workflow
  ✓ Streaming en temps réel
  ✓ Production systems
  ✓ Logique complexe
  ✓ RAG agents
""")

## 🎯 Cas d'usage LangGraph

In [ ]:
print("""
🎯 CAS D'USAGE LANGGRAPH
="*70

1️⃣ AGENTIC RAG (Advanced Retrieval-Augmented Generation)
   • Récupérer documents
   • Analyser et refiner la requête
   • Itérer jusqu'à trouver la réponse
   • HITL pour approuver la réponse

2️⃣ MULTI-AGENT WORKFLOWS
   • Agent 1: Analyze
   • Agent 2: Review
   • Agent 3: Refine
   • Tous coordonnés via LangGraph

3️⃣ HUMAN-IN-THE-LOOP SYSTEMS
   • Agent propose action
   • Humain approuve/rejette/modifie
   • Agent exécute ou reprend
   • Audit trail complet

4️⃣ STREAMING CHAT APPLICATIONS
   • Real-time token streaming
   • Multi-step reasoning visible
   • Progressive results display

5️⃣ COMPLEX DECISION WORKFLOWS
   • Multiple decision points
   • Conditional branching
   • Loop until condition met
   • State management between steps

6️⃣ PRODUCTION AI SYSTEMS
   • Checkpointing & persistence
   • Resume from interruption
   • Fork & restart
   • Full audit trail
""")

## 🚀 Points clés

In [ ]:
print("""
🚀 POINTS CLÉS LANGGRAPH
="*70

✅ STRUCTURE DE BASE:
   • StateGraph: Conteneur du graphe
   • Nodes: Fonctions qui traitent l'état
   • Edges: Connexions entre nodes
   • START/END: Points d'entrée/sortie

✅ ÉTAT (State):
   • TypedDict pour typage
   • Annotated pour réducers
   • add pour fusionner listes
   • Reducer: Règle de fusion

✅ TOOLS:
   • @tool decorator
   • bind_tools() au modèle
   • LLM décide d'utiliser
   • Tool node exécute

✅ CONTRÔLE DU FLUX:
   • add_edge(): connexion simple
   • add_conditional_edges(): routing
   • Literal[] pour typage
   • Fonction de condition

✅ HITL:
   • interrupt(): Arrête l'exécution
   • Command(resume=...): Reprend
   • Checkpointer: Sauvegarde
   • Thread ID: Identifie session

✅ STREAMING:
   • stream_mode="updates"
   • stream_mode="messages"
   • Real-time results
   • Token-level granularity

✅ PRODUCTION:
   • InMemorySaver checkpointer
   • get_state_history()
   • update_state() fork
   • Full audit trail
""")